# 🗺️ Google Maps Route Optimization — Graph Algorithms for Navigation

**Session 2 | Industrial Use Case 3 | DSA & ML for Business**

---

### Business Context
- Google Maps processes **1 billion+ km** of roads daily
- **1 billion+ users** depend on it for routing decisions
- A **0.1% improvement** in pathfinding saved users **100 million hours** of driving in 2023
- FedEx uses graph shortest-path algorithms to save **$2B+ per year** in fuel costs

### What You'll Learn
1. Build a **city road network graph** with distance, time, traffic, and toll weights
2. Compute **shortest path (Dijkstra)** optimizing for distance vs time vs cost
3. Find the **Minimum Spanning Tree** — minimum infrastructure to connect all locations
4. Identify **critical intersections** using betweenness centrality
5. **Simulate road closure** and measure rerouting impact

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv("../datasets/google_maps_routing.csv")
print(f"Road network: {len(df)} road segments")
print(f"Locations: {len(set(df['source']) | set(df['destination']))}")
print(f"Road types: {df['road_type'].value_counts().to_dict()}")
df.head()

## Step 2: Build Road Network Graph

In [ ]:
# Build undirected weighted graph (roads go both ways)
G = nx.Graph()
for _, row in df.iterrows():
    travel_time = row['base_time_min'] * row['traffic_multiplier']
    G.add_edge(row['source'], row['destination'],
               distance=row['distance_km'], time=round(travel_time, 1),
               toll=row['toll_cost'], risk=row['accident_risk'],
               road_type=row['road_type'])

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Connected: {nx.is_connected(G)}")

# Visualize
fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, seed=42, k=2)
road_colors = {'highway': '#EF4444', 'arterial': '#F59E0B', 'local': '#10B981', 'expressway': '#2563EB'}
edge_colors = [road_colors.get(G[u][v]['road_type'], '#999') for u, v in G.edges()]
edge_widths = [G[u][v]['distance'] / 5 for u, v in G.edges()]

nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths, alpha=0.6, ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=500, node_color='#1E3A5F', edgecolors='white', linewidths=2, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', font_color='white', ax=ax)

patches = [mpatches.Patch(color=c, label=t) for t, c in road_colors.items()]
ax.legend(handles=patches, loc='upper left', fontsize=10)
ax.set_title('City Road Network (Edge width = distance)', fontsize=14, fontweight='bold')
ax.axis('off'); plt.tight_layout(); plt.show()

## Step 3: Dijkstra Shortest Paths — Distance vs Time vs Cost

In [ ]:
# Compare routes optimized for different objectives
origin, destination = 'Home', 'Airport'

print(f"=== Routes from {origin} → {destination} ===\n")
for weight_name, weight in [('Distance (km)', 'distance'), ('Travel Time (min)', 'time'), ('Toll Cost ($)', 'toll')]:
    try:
        path = nx.dijkstra_path(G, origin, destination, weight=weight)
        length = nx.dijkstra_path_length(G, origin, destination, weight=weight)
        # Compute all metrics for this path
        total_dist = sum(G[path[i]][path[i+1]]['distance'] for i in range(len(path)-1))
        total_time = sum(G[path[i]][path[i+1]]['time'] for i in range(len(path)-1))
        total_toll = sum(G[path[i]][path[i+1]]['toll'] for i in range(len(path)-1))
        print(f"Optimized for: {weight_name}")
        print(f"  Path: {' → '.join(path)}")
        print(f"  Distance: {total_dist:.1f} km | Time: {total_time:.1f} min | Tolls: ${total_toll:.0f}")
        print()
    except nx.NetworkXNoPath:
        print(f"No path found for {weight_name}\n")

# BFS: find minimum-hop path
try:
    bfs_path = nx.shortest_path(G, origin, destination)
    print(f"BFS (min hops): {' → '.join(bfs_path)} ({len(bfs_path)-1} hops)")
except nx.NetworkXNoPath:
    print("No BFS path found")

## Step 4: Centrality & Road Closure Simulation

In [ ]:
# Betweenness centrality — which intersections handle the most through-traffic?
betweenness = nx.betweenness_centrality(G, weight='distance')
sorted_bc = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)

print("=== Most Critical Intersections (Betweenness Centrality) ===")
for node, bc in sorted_bc[:8]:
    print(f"  {node:<25s} | Centrality: {bc:.4f} | Degree: {G.degree(node)}")

# MST
mst = nx.minimum_spanning_tree(G, weight='distance')
mst_dist = sum(d['distance'] for _, _, d in mst.edges(data=True))
total_dist = sum(d['distance'] for _, _, d in G.edges(data=True))
print(f"\nMinimum Spanning Tree: {mst_dist:.1f} km (vs {total_dist:.1f} km total)")
print(f"Savings: {(1-mst_dist/total_dist)*100:.0f}% of road network is redundancy")

# Road closure simulation
critical_node = sorted_bc[0][0]
print(f"\n=== ROAD CLOSURE: Removing '{critical_node}' ===")
G_closed = G.copy(); G_closed.remove_node(critical_node)
print(f"Connected after removal: {nx.is_connected(G_closed)}")
if not nx.is_connected(G_closed):
    components = list(nx.connected_components(G_closed))
    print(f"Network splits into {len(components)} disconnected zones")
    for i, comp in enumerate(components):
        print(f"  Zone {i+1}: {', '.join(sorted(comp)[:5])}{'...' if len(comp)>5 else ''} ({len(comp)} locations)")
else:
    # Check detour impact
    pairs = [('Home', 'Airport'), ('Office', 'Stadium'), ('University', 'Hospital')]
    for src, dst in pairs:
        if src == critical_node or dst == critical_node: continue
        try:
            old = nx.dijkstra_path_length(G, src, dst, weight='time')
            new = nx.dijkstra_path_length(G_closed, src, dst, weight='time')
            print(f"  {src}→{dst}: {old:.1f} min → {new:.1f} min (+{new-old:.1f} min detour)")
        except: pass